# Laboratorio Semana 2 - T-SQL en AdventureWorks

Este notebook es para resolver ejercicios practicos de T-SQL usando la base de datos AdventureWorks.
Avanzaremos paso a paso juntos, probando cada consulta y validando resultados.

In [41]:
-- Detalle: tablas columna por columna
SELECT 
    c.TABLE_SCHEMA,
    c.TABLE_NAME,
    c.COLUMN_NAME,
    c.DATA_TYPE
FROM INFORMATION_SCHEMA.COLUMNS AS c
WHERE c.TABLE_SCHEMA IN ('adventureworks', 'dbo')
  AND c.TABLE_NAME LIKE 'adventureworks_%'
ORDER BY c.TABLE_SCHEMA, c.TABLE_NAME, c.ORDINAL_POSITION;

(62 rows affected)

TABLE_SCHEMA   | TABLE_NAME                           | COLUMN_NAME           | DATA_TYPE
---------------+--------------------------------------+-----------------------+----------
adventureworks | adventureworks_calendar              | Date                  | varchar  
adventureworks | adventureworks_customers             | CustomerKey           | int      
adventureworks | adventureworks_customers             | Prefix                | varchar  
adventureworks | adventureworks_customers             | FirstName             | varchar  
adventureworks | adventureworks_customers             | LastName              | varchar  
adventureworks | adventureworks_customers             | BirthDate             | varchar  
adventureworks | adventureworks_customers             | MaritalStatus         | varchar  
adventureworks | adventureworks_customers             | Gender                | varchar  
adventureworks | adventureworks_customers             | EmailAddress          | varchar  
adventurew

## Paso siguiente: mover tablas a WAREHOUSE

Ejecuta la siguiente celda conectado a la conexion WAREHOUSE (base de datos SEMANA2_WAREHOUSE).

In [42]:
-- Debes ejecutar esta celda conectado a: SEMANA2_WAREHOUSE
-- Origen: SEMANA2 (SQL Analytics Endpoint)

IF DB_NAME() <> 'SEMANA2_WAREHOUSE'
BEGIN
    SELECT
        DB_NAME() AS CurrentDatabase,
        'Error conectar a BD correspondiente' AS WarningMessage;
    RETURN;
END;

IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'adventureworks')
BEGIN
    EXEC('CREATE SCHEMA adventureworks');
END;

IF OBJECT_ID('adventureworks.adventureworks_calendar', 'U') IS NULL
    CREATE TABLE adventureworks.adventureworks_calendar AS
    SELECT * FROM [SEMANA2].[adventureworks].[adventureworks_calendar];

IF OBJECT_ID('adventureworks.adventureworks_customers', 'U') IS NULL
    CREATE TABLE adventureworks.adventureworks_customers AS
    SELECT * FROM [SEMANA2].[adventureworks].[adventureworks_customers];

IF OBJECT_ID('adventureworks.adventureworks_product_categories', 'U') IS NULL
    CREATE TABLE adventureworks.adventureworks_product_categories AS
    SELECT * FROM [SEMANA2].[adventureworks].[adventureworks_product_categories];

IF OBJECT_ID('adventureworks.adventureworks_product_subcategories', 'U') IS NULL
    CREATE TABLE adventureworks.adventureworks_product_subcategories AS
    SELECT * FROM [SEMANA2].[adventureworks].[adventureworks_product_subcategories];

IF OBJECT_ID('adventureworks.adventureworks_products', 'U') IS NULL
    CREATE TABLE adventureworks.adventureworks_products AS
    SELECT * FROM [SEMANA2].[adventureworks].[adventureworks_products];

IF OBJECT_ID('adventureworks.adventureworks_returns', 'U') IS NULL
    CREATE TABLE adventureworks.adventureworks_returns AS
    SELECT * FROM [SEMANA2].[adventureworks].[adventureworks_returns];

IF OBJECT_ID('adventureworks.adventureworks_sales_2015', 'U') IS NULL
    CREATE TABLE adventureworks.adventureworks_sales_2015 AS
    SELECT * FROM [SEMANA2].[adventureworks].[adventureworks_sales_2015];

IF OBJECT_ID('adventureworks.adventureworks_sales_2016', 'U') IS NULL
    CREATE TABLE adventureworks.adventureworks_sales_2016 AS
    SELECT * FROM [SEMANA2].[adventureworks].[adventureworks_sales_2016];

IF OBJECT_ID('adventureworks.adventureworks_sales_2017', 'U') IS NULL
    CREATE TABLE adventureworks.adventureworks_sales_2017 AS
    SELECT * FROM [SEMANA2].[dbo].[adventureworks_sales_2017];

IF OBJECT_ID('adventureworks.adventureworks_territories', 'U') IS NULL
    CREATE TABLE adventureworks.adventureworks_territories AS
    SELECT * FROM [SEMANA2].[adventureworks].[adventureworks_territories];

Commands completed successfully.

In [43]:
-- Verificacion final en WAREHOUSE
SELECT
    TABLE_SCHEMA,
    TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'adventureworks'
ORDER BY TABLE_NAME;

(11 rows affected)

TABLE_SCHEMA   | TABLE_NAME                          
---------------+-------------------------------------
adventureworks | adventureworks_calendar             
adventureworks | adventureworks_customers            
adventureworks | adventureworks_product_categories   
adventureworks | adventureworks_product_subcategories
adventureworks | adventureworks_products             
adventureworks | adventureworks_returns              
adventureworks | adventureworks_sales_2015           
adventureworks | adventureworks_sales_2016           
adventureworks | adventureworks_sales_2017           
adventureworks | adventureworks_territories          
adventureworks | sales_historico                     
(11 rows)

## Crear tabla historica de ventas

En este paso crearemos `adventureworks.sales_historico` en el WAREHOUSE.
La tabla se crea copiando la estructura (tipos de dato) de `adventureworks.adventureworks_sales_2017`, sin cargar datos todavia.

## Validar conexion y existencia de tabla origen

In [45]:
-- Preparar recreacion: eliminar tabla historica si ya existe
IF OBJECT_ID('[adventureworks].[sales_historico]', 'U') IS NOT NULL
BEGIN
    DROP TABLE [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico];
END;

Commands completed successfully.

## Comprobar tabla origen de ventas

In [46]:
SELECT
    TABLE_CATALOG,
    TABLE_SCHEMA,
    TABLE_NAME
FROM [SEMANA2_WAREHOUSE].INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'adventureworks'
  AND TABLE_NAME = 'adventureworks_sales_2017';

(1 row affected)

TABLE_CATALOG     | TABLE_SCHEMA   | TABLE_NAME               
------------------+----------------+--------------------------
SEMANA2_WAREHOUSE | adventureworks | adventureworks_sales_2017
(1 row)

##  Crear tabla historica sin datos (CTAS)

In [47]:
CREATE TABLE [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico] AS
SELECT *
FROM [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_sales_2017]
WHERE 1 = 0;

Statement ID: {AE5E178A-2DD7-4F43-A127-BAD2A5DF1DA4} | Query hash: 0x659297846C7728E | Distributed request ID: {8263E17F-1EDE-4B30-8269-1213AF4E4AF8}
(0 rows affected)

## Validar que la tabla historica quedo vacia

In [48]:
SELECT COUNT(*) AS RowCountHistorico
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico];

Statement ID: {659FB838-54A4-419A-A884-874BCCFF63DD} | Query hash: 0x91ADB7216E3F20D3 | Distributed request ID: {3DC9ED84-1928-486C-B3D3-4E4880DAF087}
(1 row affected)

RowCountHistorico
-----------------
0                
(1 row)

## 3. DML - Data Manipulation Language

Inserta en una sola corrida los datos de ventas 2015, 2016 y 2017 en `adventureworks.sales_historico`.

##  Insert

In [49]:
INSERT INTO [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
SELECT * FROM [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_sales_2015]
UNION ALL
SELECT * FROM [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_sales_2016]
UNION ALL
SELECT * FROM [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_sales_2017];

Statement ID: {F9E19B66-2ED6-4B7B-853B-001E136AAD18} | Query hash: 0xA43CB5B3767FE2CC | Distributed request ID: {5B4B03CE-DD52-44DF-BF5C-7752372342F5}
(56046 rows affected)

## Update

Ejemplo de `UPDATE` en el mismo estilo del cheat sheet: actualiza una columna con condiciones en `WHERE`.

In [55]:
-- Revisar muestra antes del update
SELECT  TOP 20 *
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
ORDER BY TerritoryKey, OrderLineItem;

Statement ID: {CBA9965B-1F40-4EC3-8C22-37840E8D3ECD} | Query hash: 0x223CD04FC3FB2CFF | Distributed request ID: {7C7DDD18-199D-4280-B15F-DE00421A9C06}
(20 rows affected)

OrderDate | StockDate | OrderNumber | ProductKey | CustomerKey | TerritoryKey | OrderLineItem | OrderQuantity
----------+-----------+-------------+------------+-------------+--------------+---------------+--------------
8/2/2016  | 4/7/2003  | SO51939     | 536        | 23670       | 1            | 1             | 3            
8/5/2016  | 5/13/2003 | SO52082     | 528        | 23831       | 1            | 1             | 3            
8/18/2016 | 5/16/2003 | SO52737     | 529        | 24389       | 1            | 1             | 3            
8/7/2016  | 4/16/2003 | SO52189     | 528        | 15257       | 1            | 1             | 3            
8/8/2016  | 6/25/2003 | SO52236     | 536        | 23050       | 1            | 1             | 3            
8/20/2016 | 5/3/2003  | SO52874     | 485        | 12906       | 1            | 1             | 3            
8/24/2016 | 7/26/2003 | SO53022     | 530        | 28003       | 1            | 1             | 3            
8/16/2016 

In [56]:
-- Update estilo cheat sheet (SET + 2 condiciones en WHERE)
UPDATE [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
SET OrderQuantity = OrderQuantity + 1
WHERE TerritoryKey = 1
  AND OrderQuantity > 0;

Statement ID: {7741D567-F583-4A94-A6C3-3A708708286B} | Query hash: 0xA975FABD94E01422 | Distributed request ID: {1221B17D-8F79-4CD0-87FD-2265EF55626E}
(8267 rows affected)

In [59]:
-- Revisar muestra despues del update
SELECT TOP 20
    TerritoryKey,
    OrderLineItem,
    OrderQuantity
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
ORDER BY TerritoryKey, OrderLineItem;

Statement ID: {C24A3D71-4CBC-46FB-9ACA-68B006D7D85A} | Query hash: 0x9602679FFD533E39 | Distributed request ID: {AD96968A-540A-405D-9CB9-441A2FC948DD}
(20 rows affected)

TerritoryKey | OrderLineItem | OrderQuantity
-------------+---------------+--------------
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
1            | 1             | 4            
(20 rows)

## MERGE (patron Upsert)

Este ejemplo compara la tabla historica (`target`) contra ventas 2017 (`source`) usando la clave compuesta `OrderNumber + OrderLineItem`.

- Si existe en ambas: hace `UPDATE`.
- Si solo existe en source: hace `INSERT`.
- Si solo existe en target: en este ejemplo **no borra** (para evitar perdidas accidentales).


In [58]:
MERGE INTO [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico] AS target
USING [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_sales_2017] AS source
   ON target.OrderNumber = source.OrderNumber
  AND target.OrderLineItem = source.OrderLineItem

WHEN MATCHED THEN
    UPDATE SET
        target.OrderQuantity = source.OrderQuantity,
        target.StockDate = source.StockDate

WHEN NOT MATCHED BY TARGET THEN
    INSERT (
        OrderDate,
        StockDate,
        OrderNumber,
        ProductKey,
        CustomerKey,
        TerritoryKey,
        OrderLineItem,
        OrderQuantity
    )
    VALUES (
        source.OrderDate,
        source.StockDate,
        source.OrderNumber,
        source.ProductKey,
        source.CustomerKey,
        source.TerritoryKey,
        source.OrderLineItem,
        source.OrderQuantity
    );

Statement ID: {2918550C-BE63-4989-A4DE-328C9739680F} | Query hash: 0xA7ADB85917EA2F9D | Distributed request ID: {1804B8C8-BE4E-4CDA-BFFC-5D597185B438}
(29481 rows affected)

Tip de examen (mas detallado):

En DP-600, `MERGE` es clave para cargas incrementales y sincronizacion de capas. Lo importante no es memorizar la sintaxis completa, sino dominar la logica de matching:

1. Define bien la clave en `ON` (si la clave es mala, duplicas o sobreescribes mal).
2. Usa `WHEN MATCHED` para cambios de registros existentes.
3. Usa `WHEN NOT MATCHED BY TARGET` para nuevos registros.
4. Usa `WHEN NOT MATCHED BY SOURCE` solo cuando realmente quieras eliminar historico (normalmente se evita en tablas historicas).
5. En cargas reales, primero prueba con `SELECT` del join para validar cuantas filas caeran en cada caso antes de ejecutar el `MERGE`.

## SELECT INTO

In [1]:
-- Crea una tabla nueva a partir de resultados de consulta (resumen por cliente)
SELECT
    c.[CustomerKey] AS [customer_id],
    COUNT(*) AS [order_count],
    SUM(s.[OrderQuantity]) AS [lifetime_value]
INTO [SEMANA2_WAREHOUSE].[adventureworks].[customer_summary]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico] AS s
INNER JOIN [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_customers] AS c
    ON c.[CustomerKey] = s.[CustomerKey]
GROUP BY c.[CustomerKey];

Statement ID: {E2B43A99-3B5F-4E5C-82A5-D7E6A94F0F73} | Query hash: 0xAEEE604049ACCFDF | Distributed request ID: {D9C4B0AA-9F92-447C-B2A7-724D60AE1CB3}
(17416 rows affected)

## 4. Fundamentos de consulta


-- SELECT, WHERE, ORDER BY

In [23]:
-- SELECT, WHERE, ORDER BY
SELECT TOP 20
    [OrderNumber] AS [order_id],
    [CustomerKey] AS [customer_id],
    [OrderDate] AS [order_date],
    [OrderQuantity] AS [total_amount]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
WHERE [OrderQuantity] > 1
  AND [OrderDate] >= '2017-01-01'
ORDER BY [OrderDate] DESC, [OrderQuantity] ASC;

Statement ID: {7433B3A7-0E2E-48B7-852E-B1DFEE6268CB} | Query hash: 0x7E6357D21883F0DA | Distributed request ID: {59CAD8CF-94FA-424D-B0A0-A5BD2ABBAE48}
(20 rows affected)

order_id | customer_id | order_date | total_amount
---------+-------------+------------+-------------
SO74135  | 28368       | 2017-06-30 | 2           
SO74107  | 13570       | 2017-06-30 | 2           
SO74129  | 18400       | 2017-06-30 | 2           
SO74108  | 14984       | 2017-06-30 | 2           
SO74129  | 18400       | 2017-06-30 | 2           
SO74114  | 21495       | 2017-06-30 | 2           
SO74118  | 23513       | 2017-06-30 | 2           
SO74143  | 28517       | 2017-06-30 | 2           
SO74104  | 20807       | 2017-06-30 | 2           
SO74143  | 28517       | 2017-06-30 | 2           
SO74108  | 14984       | 2017-06-30 | 2           
SO74123  | 24571       | 2017-06-30 | 2           
SO74120  | 17967       | 2017-06-30 | 2           
SO74100  | 21992       | 2017-06-30 | 2           
SO74137  | 22202       | 2017-06-30 | 2           
SO74124  | 21676       | 2017-06-30 | 2           
SO74100  | 21992       | 2017-06-30 | 2           
SO74104  | 20807       | 2017-0

-- DISTINCT y TOP

In [1]:
-- DISTINCT y TOP (usando tabla de devoluciones, no ventas)
SELECT DISTINCT TOP 10 [ProductKey] AS [product_id]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_returns];

-- TOP 10 por mayor cantidad de devoluciones
SELECT TOP 10 *
FROM [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_returns]
ORDER BY [ReturnQuantity] DESC;

-- TOP 5 con empates
SELECT TOP 5 WITH TIES *
FROM [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_returns]
ORDER BY [ReturnQuantity] DESC;

Statement ID: {BACB65D1-EA27-440C-A0FE-BC35648F33C2} | Query hash: 0xAE10DA679D6BFE | Distributed request ID: {A2BEB097-28A7-40B1-AAE3-C881E40FFD05}
(10 rows affected)
Statement ID: {0E90B7C6-22FA-491A-94C8-E8E35397B5D7} | Query hash: 0x28F5FAED260222ED | Distributed request ID: {8B741907-4DA1-4C32-8D23-DE3FF23ED0D5}
(10 rows affected)
Statement ID: {98B4E3AA-3660-4198-8E82-0AF2608A74B3} | Query hash: 0x692912B97655DF00 | Distributed request ID: {A9A3FDF4-672E-4196-8788-28F47B437E4D}
(19 rows affected)

product_id
----------
476       
569       
587       
340       
462       
529       
563       
580       
583       
576       
(10 rows)

ReturnDate | TerritoryKey | ProductKey | ReturnQuantity
-----------+--------------+------------+---------------
10/13/2015 | 9            | 352        | 2             
8/19/2016  | 4            | 477        | 2             
8/30/2016  | 1            | 480        | 2             
9/16/2016  | 6            | 478        | 2             
9/30/2016  | 6            | 528        | 2             
11/6/2016  | 4            | 477        | 2             
11/7/2016  | 4            | 477        | 2             
1/1/2017   | 9            | 479        | 2             
1/18/2017  | 6            | 481        | 2             
1/24/2017  | 9            | 477        | 2             
(10 rows)

ReturnDate | TerritoryKey | ProductKey | ReturnQuantity
-----------+--------------+------------+---------------
10/13/2015 | 9            | 352        | 2             
8/19/2016  | 4            | 477        | 2             
8/30/2016  | 1            | 480        | 2             
9/16/2016  | 6            | 478        | 2             
9/30/2016  | 6            | 528        | 2             
11/6/2016  | 4            | 477        | 2             
11/7/2016  | 4            | 477        | 2             
1/1/2017   | 9            | 479        | 2             
1/18/2017  | 6            | 481        | 2             
1/24/2017  | 9            | 477        | 2             
2/4/2017   | 8            | 480        | 2             
2/11/2017  | 6            | 478        | 2             
2/17/2017  | 4            | 605        | 2             
2/24/2017  | 9            | 477        | 2             
3/1/2017   | 4            | 477        | 2             
3/13/2017  | 8            | 480        | 2      

-- OFFSET-FETCH (paginacion)

In [8]:
-- OFFSET-FETCH (paginacion)
SELECT TOP 10
    [OrderNumber] AS [order_id],
    [CustomerKey] AS [customer_id],
    [OrderQuantity] AS [total_amount]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
ORDER BY [OrderNumber]
OFFSET 20 ROWS
FETCH NEXT 10 ROWS ONLY;

Statement ID: {49F72C87-29B7-4251-8499-BD4F67B81C70} | Query hash: 0xD4329D6AC62E2D48 | Distributed request ID: {B2D3935A-7B38-4938-9889-3744A11AF73D}
(10 rows affected)

order_id | customer_id | total_amount
---------+-------------+-------------
SO45099  | 29174       | 2           
SO45100  | 19428       | 1           
SO45101  | 22748       | 1           
SO45102  | 29274       | 1           
SO45103  | 29140       | 2           
SO45104  | 29142       | 1           
SO45105  | 22765       | 1           
SO45106  | 19987       | 1           
SO45107  | 29275       | 1           
SO45108  | 22975       | 1           
(10 rows)

-- GROUP BY y HAVING

In [29]:
-- GROUP BY y HAVING
-- WHERE filtra filas individuales antes de la agregacion.
-- HAVING filtra resultados agrupados despues de la agregacion.
SELECT TOP 15
    [CustomerKey] AS [customer_id],
    COUNT(*) AS [order_count],
    SUM([OrderQuantity]) AS [total_spent],
    AVG(CAST([OrderQuantity] AS FLOAT)) AS [avg_order]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
WHERE [OrderDate] >= '2017-01-01'
GROUP BY [CustomerKey]
HAVING COUNT(*) >= 5
ORDER BY [total_spent] DESC;

Statement ID: {5C420086-3E17-45EC-9EE1-BD8934B5EEBD} | Query hash: 0x5BBB56823085A72E | Distributed request ID: {C776FCB5-EB9E-42A7-A328-DEE07A3629EC}
(15 rows affected)

customer_id | order_count | total_spent | avg_order         
------------+-------------+-------------+-------------------
11262       | 40          | 74          | 1,85              
11300       | 45          | 74          | 1,6444444444444444
11185       | 45          | 72          | 1,6               
11223       | 33          | 59          | 1,7878787878787878
11276       | 36          | 54          | 1,5               
11330       | 31          | 53          | 1,7096774193548387
11277       | 29          | 52          | 1,793103448275862 
11711       | 28          | 48          | 1,7142857142857142
11091       | 30          | 46          | 1,5333333333333334
11566       | 25          | 45          | 1,8               
11287       | 22          | 44          | 2                 
11331       | 22          | 40          | 1,8181818181818181
11211       | 22          | 38          | 1,7272727272727273
11200       | 23          | 35          | 1,5217391304347827
11142       | 21        

-- Expresion CASE


In [30]:
-- Expresion CASE
SELECT TOP 15
    [OrderNumber] AS [order_id],
    [OrderQuantity] AS [total_amount],
    CASE
        WHEN [OrderQuantity] >= 4 THEN 'High'
        WHEN [OrderQuantity] >= 2 THEN 'Medium'
        WHEN [OrderQuantity] = 1 THEN 'Low'
        ELSE 'Micro'
    END AS [order_tier]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico];

Statement ID: {F3460963-51C5-4902-ACF0-22A05DF99A83} | Query hash: 0xFE58FC314E073B0D | Distributed request ID: {6D99D993-3FDF-4FA6-AB22-51A0C3F77196}
(15 rows affected)

order_id | total_amount | order_tier
---------+--------------+-----------
SO58845  | 1            | Low       
SO70714  | 2            | Medium    
SO72656  | 2            | Medium    
SO51555  | 1            | Low       
SO54042  | 3            | Medium    
SO58572  | 1            | Low       
SO58845  | 1            | Low       
SO60233  | 1            | Low       
SO54784  | 2            | Medium    
SO61412  | 1            | Low       
SO62984  | 1            | Low       
SO64042  | 2            | Medium    
SO64542  | 1            | Low       
SO70714  | 1            | Low       
SO71961  | 1            | Low       
(15 rows)

## 5. JOINs

### Tipos de JOIN de un vistazo

| JOIN | Devuelve |
| --- | --- |
| INNER JOIN | Solo filas que coinciden en ambas tablas |
| LEFT JOIN | Todas las filas de la tabla izquierda + coincidencias de la derecha |
| RIGHT JOIN | Todas las filas de la tabla derecha + coincidencias de la izquierda |
| FULL OUTER JOIN | Todas las filas de ambas tablas, `NULL` donde no hay coincidencia |
| CROSS JOIN | Producto cartesiano: cada fila por cada fila |

**Nota:** en este laboratorio usamos tablas de `adventureworks` en `SEMANA2_WAREHOUSE`.

### INNER JOIN

Devuelve solo filas que tienen coincidencia en ambas tablas.
Aqui usamos ventas historicas + clientes por `CustomerKey`.

In [2]:
-- INNER JOIN: solo filas con coincidencia en ambas tablas
SELECT TOP 20
    s.[OrderNumber] AS [order_id],
    c.[CustomerKey] AS [customer_id],
    s.[OrderQuantity] AS [total_amount]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico] AS s
INNER JOIN [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_customers] AS c
    ON s.[CustomerKey] = c.[CustomerKey]
ORDER BY s.[OrderNumber];

Statement ID: {2E164712-07E6-4875-93D4-38C860CDBD74} | Query hash: 0xAEEEF7113FD440DD | Distributed request ID: {2490423A-4B5D-4B13-B319-90FE21E4EE6C}
(20 rows affected)

order_id | customer_id | total_amount
---------+-------------+-------------
SO45079  | 29255       | 1           
SO45080  | 14657       | 2           
SO45081  | 26782       | 1           
SO45082  | 11455       | 1           
SO45083  | 14947       | 1           
SO45084  | 29143       | 1           
SO45085  | 18746       | 1           
SO45086  | 18747       | 1           
SO45087  | 11388       | 1           
SO45088  | 11398       | 1           
SO45089  | 25977       | 1           
SO45090  | 29170       | 1           
SO45091  | 18909       | 1           
SO45092  | 18899       | 1           
SO45093  | 18906       | 1           
SO45094  | 22785       | 1           
SO45095  | 11394       | 1           
SO45096  | 12483       | 1           
SO45097  | 29151       | 1           
SO45098  | 29167       | 2           
(20 rows)

### LEFT JOIN

Devuelve todas las filas de la tabla izquierda, incluso si no hay coincidencia en la derecha.
Aqui usamos todos los clientes y sumamos su cantidad de pedidos si existe.

In [3]:
-- LEFT JOIN: todos los clientes, tengan o no pedidos
SELECT TOP 20
    c.[CustomerKey] AS [customer_id],
    COALESCE(SUM(s.[OrderQuantity]), 0) AS [total_spent]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_customers] AS c
LEFT JOIN [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico] AS s
    ON c.[CustomerKey] = s.[CustomerKey]
GROUP BY c.[CustomerKey]
ORDER BY [total_spent] DESC, [customer_id];

Statement ID: {DD5FDDA0-B48C-4288-8C9F-978DC953F376} | Query hash: 0x6601C77E223D5254 | Distributed request ID: {454A31C8-2104-4C90-87B7-A24A14FBC0EC}
(20 rows affected)

customer_id | total_spent
------------+------------
11262       | 106        
11300       | 106        
11331       | 102        
11185       | 100        
11566       | 99         
11091       | 97         
11277       | 97         
11287       | 97         
11330       | 97         
11223       | 90         
11711       | 89         
11176       | 88         
11200       | 83         
11276       | 76         
11505       | 61         
11631       | 58         
11506       | 56         
11520       | 55         
11211       | 54         
11502       | 54         
(20 rows)

### FULL OUTER JOIN

Devuelve todas las filas de ambas tablas, con `NULL` cuando no hay coincidencia.
Aqui comparamos cantidades por producto entre devoluciones y ventas historicas.

In [6]:
-- FULL OUTER JOIN: productos con o sin coincidencia en ambas fuentes
WITH warehouse_returns AS (
    SELECT
        [ProductKey],
        SUM([ReturnQuantity]) AS [warehouse_qty]
    FROM [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_returns]
    GROUP BY [ProductKey]
),
store_sales AS (
    SELECT
        [ProductKey],
        SUM([OrderQuantity]) AS [store_qty]
    FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
    GROUP BY [ProductKey]
)
SELECT TOP 50
    COALESCE(a.[ProductKey], b.[ProductKey]) AS [product_id],
    a.[warehouse_qty],
    b.[store_qty]
FROM warehouse_returns AS a
FULL OUTER JOIN store_sales AS b
    ON a.[ProductKey] = b.[ProductKey]
ORDER BY [product_id];

Statement ID: {725414D8-43C3-4405-98BA-06B6EC3D2EE6} | Query hash: 0x971CDF3B132C4450 | Distributed request ID: {1673A221-43E8-4DEC-AA93-1E33B0466BE4}
(50 rows affected)

product_id | warehouse_qty | store_qty
-----------+---------------+----------
214        | 70            | 2204     
215        | 52            | 2071     
220        | 66            | 2130     
223        | 46            | 4257     
226        | 12            | 417      
229        | 15            | 432      
232        | 15            | 456      
235        | 10            | 419      
310        | 4             | 201      
311        | 7             | 155      
312        | 8             | 203      
313        | 5             | 191      
314        | 5             | 181      
320        | 3             | 75       
322        | 2             | 45       
324        | 3             | 81       
326        | 3             | 70       
328        | 4             | 88       
330        | 6             | 59       
332        | 2             | 74       
334        | 2             | 68       
336        | 1             | 58       
338        | NULL          | 55       
340        | 1           

### CROSS JOIN

Genera el producto cartesiano: cada fila de la primera tabla con cada fila de la segunda.
Aqui generamos una combinacion por cada producto y cada cliente.

In [7]:
-- CROSS JOIN: una fila por cada producto por cada cliente
SELECT TOP 50
    p.[ProductKey] AS [product_id],
    c.[CustomerKey] AS [customer_id]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_products] AS p
CROSS JOIN [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_customers] AS c
ORDER BY p.[ProductKey], c.[CustomerKey];

Statement ID: {1B275A1C-BF7E-4B85-8B97-1A78009A7DCA} | Query hash: 0x66F54008DF50456E | Distributed request ID: {E4EB5F06-979C-4EE4-9758-9DBEBD350E42}
(50 rows affected)

product_id | customer_id
-----------+------------
214        | 11000      
214        | 11001      
214        | 11002      
214        | 11003      
214        | 11004      
214        | 11005      
214        | 11007      
214        | 11008      
214        | 11009      
214        | 11010      
214        | 11011      
214        | 11012      
214        | 11013      
214        | 11014      
214        | 11015      
214        | 11016      
214        | 11017      
214        | 11018      
214        | 11019      
214        | 11020      
214        | 11021      
214        | 11022      
214        | 11023      
214        | 11024      
214        | 11025      
214        | 11026      
214        | 11027      
214        | 11028      
214        | 11029      
214        | 11030      
214        | 11031      
214        | 11032      
214        | 11033      
214        | 11034      
214        | 11035      
214        | 11036      
214        | 11037      
214        | 11038      


### Self Join

Un self join une una tabla consigo misma usando dos alias distintos.
Aqui buscamos pares de pedidos del mismo cliente (pedido actual vs pedido previo).

In [8]:
-- SELF JOIN: pedidos del mismo cliente en diferentes fechas
SELECT TOP 30
    a.[CustomerKey] AS [customer_id],
    a.[OrderNumber] AS [current_order_id],
    a.[OrderDate] AS [current_order_date],
    b.[OrderNumber] AS [previous_order_id],
    b.[OrderDate] AS [previous_order_date]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico] AS a
LEFT JOIN [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico] AS b
    ON a.[CustomerKey] = b.[CustomerKey]
   AND b.[OrderDate] < a.[OrderDate]
   AND a.[OrderNumber] <> b.[OrderNumber]
ORDER BY a.[CustomerKey], a.[OrderDate];

Statement ID: {2E44E276-C448-4C48-80B4-352F01A0912D} | Query hash: 0x8C691FAA4BA2907 | Distributed request ID: {2A6A84AC-BE9C-4563-8760-6D85D7601789}
(30 rows affected)

customer_id | current_order_id | current_order_date | previous_order_id | previous_order_date
------------+------------------+--------------------+-------------------+--------------------
11000       | SO51522          | 2016-07-22         | NULL              | NULL               
11000       | SO51522          | 2016-07-22         | NULL              | NULL               
11000       | SO57418          | 2016-11-04         | SO51522           | 2016-07-22         
11000       | SO57418          | 2016-11-04         | SO51522           | 2016-07-22         
11000       | SO57418          | 2016-11-04         | SO51522           | 2016-07-22         
11000       | SO57418          | 2016-11-04         | SO51522           | 2016-07-22         
11000       | SO57418          | 2016-11-04         | SO51522           | 2016-07-22         
11000       | SO57418          | 2016-11-04         | SO51522           | 2016-07-22         
11000       | SO57418          | 2016-11-04         | SO5152

### Exam tip

Conoce como identificar que tipo de JOIN produce el resultado correcto segun el escenario.
`LEFT JOIN` con filtro sobre la tabla derecha en `WHERE` (por ejemplo, `WHERE b.id IS NULL`) permite encontrar filas sin coincidencia.

## 6. Funciones de agregacion

Para esta seccion, la tabla mas util es `adventureworks.sales_historico` porque contiene columnas clave para agregaciones: `CustomerKey`, `ProductKey`, `OrderQuantity` y `OrderDate`.

| Funcion | Descripcion | Ejemplo adaptado |
| --- | --- | --- |
| `COUNT(*)` | Contar todas las filas (incluye NULLs) | `COUNT(*)` |
| `COUNT(col)` | Contar valores no NULL | `COUNT([OrderDate])` |
| `COUNT(DISTINCT col)` | Contar valores unicos no NULL | `COUNT(DISTINCT [CustomerKey])` |
| `SUM(col)` | Total de columna numerica | `SUM([OrderQuantity])` |
| `AVG(col)` | Promedio (ignora NULLs) | `AVG(CAST([OrderQuantity] AS FLOAT))` |
| `MIN(col)` | Valor minimo | `MIN([OrderDate])` |
| `MAX(col)` | Valor maximo | `MAX([OrderQuantity])` |
| `STRING_AGG(col, sep)` | Concatenar con separador | `STRING_AGG(CAST([ProductKey] AS VARCHAR(20)), ', ')` |

In [9]:
-- COUNT(*): total de filas
SELECT COUNT(*) AS [total_rows]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico];

-- COUNT(col): cuenta no NULL
SELECT COUNT([OrderDate]) AS [order_date_not_null]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico];

-- COUNT(DISTINCT col): clientes unicos
SELECT COUNT(DISTINCT [CustomerKey]) AS [unique_customers]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico];

-- SUM(col): suma total de cantidad
SELECT SUM([OrderQuantity]) AS [total_quantity]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico];

-- AVG(col): promedio de cantidad
SELECT AVG(CAST([OrderQuantity] AS FLOAT)) AS [avg_quantity]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico];

-- MIN(col): fecha minima de pedido
SELECT MIN([OrderDate]) AS [min_order_date]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico];

-- MAX(col): cantidad maxima en una linea
SELECT MAX([OrderQuantity]) AS [max_order_quantity]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico];

Statement ID: {E82C9873-ADEA-426D-A02A-20DFCEDECED9} | Query hash: 0x91ADB7216E3F20D3 | Distributed request ID: {066E9D7E-A652-42F2-A54D-F32B017D1A04}
(1 row affected)
Statement ID: {AD8C4C0E-87EA-460A-9EFC-465D35707280} | Query hash: 0x9242D6FAA75C76EC | Distributed request ID: {2822296D-2265-42F1-ACA2-DC09785264D5}
(1 row affected)
Statement ID: {62B407B7-B492-4640-B734-2CCC5249E20E} | Query hash: 0x6A85DC2C20DDD9C6 | Distributed request ID: {4680A278-3A1F-47A1-905C-4C6C2A20A4EB}
(1 row affected)
Statement ID: {192C0E14-6A4B-42AF-9182-A4EB79D1A12C} | Query hash: 0xC7F517233BAA00C5 | Distributed request ID: {0062CB76-7908-4F01-B5D9-AE01167BB372}
(1 row affected)
Statement ID: {D4C01022-CA30-4D4F-A199-054468E31725} | Query hash: 0x4CB84ADCF22B6752 | Distributed request ID: {A58CC048-D3F9-4D06-92F7-11FF39F46B29}
(1 row affected)
Statement ID: {A1358439-5082-49AF-B2EC-3654020E6BF5} | Query hash: 0xB01C4609EC3EB544 | Distributed request ID: {FFE571D9-5943-4C82-B5D1-FB1C834B417A}
(1 row af

total_rows
----------
56046     
(1 row)

order_date_not_null
-------------------
56046              
(1 row)

unique_customers
----------------
17416           
(1 row)

total_quantity
--------------
88021         
(1 row)

avg_quantity      
------------------
1,5705135067623024
(1 row)

min_order_date
--------------
2015-01-01    
(1 row)

max_order_quantity
------------------
4                 
(1 row)

### Ejemplo de STRING_AGG

En este laboratorio, en lugar de `product_name` usamos `ProductKey` (disponible y consistente en `sales_historico`).

In [10]:
SELECT TOP 50
    [CustomerKey] AS [customer_id],
    STRING_AGG(CAST([ProductKey] AS VARCHAR(20)), ', ')
        WITHIN GROUP (ORDER BY [ProductKey]) AS [products_ordered]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
GROUP BY [CustomerKey]
ORDER BY [customer_id];

Statement ID: {2561764F-4448-43C6-B5C4-E4C49C3297A5} | Query hash: 0xEB32F94644FCADE2 | Distributed request ID: {C3B00B1A-1FEF-40AE-84FD-97FB6B7B08A0}
(50 rows affected)

customer_id | products_ordered                                                                                                                               
------------+------------------------------------------------------------------------------------------------------------------------------------------------
11000       | 214, 352, 485, 488, 530, 541, 573                                                                                                              
11001       | 215, 223, 352, 477, 477, 478, 479, 485, 491, 604                                                                                               
11002       | 220, 358, 561                                                                                                                                  
11003       | 223, 360, 477, 478, 480, 530, 541, 564                                                                                                         
11004       | 214, 215, 354, 485, 562               

## 7. Window Functions

Regla clave: las window functions calculan un valor para cada fila en relacion con una ventana de filas, sin colapsar el resultado como lo hace `GROUP BY`.

Patron de sintaxis:

```sql
FUNCTION_NAME(args) OVER (
    [PARTITION BY col1, col2]
    [ORDER BY col3 ASC|DESC]
    [ROWS|RANGE BETWEEN ... AND ...]
 )
```

### Ranking Functions

| Funcion | Empates | Huecos | Descripcion |
| --- | --- | --- | --- |
| `ROW_NUMBER()` | No maneja empates | - | Numero secuencial unico por particion |
| `RANK()` | Mismo rank para empates | Si | Deja huecos despues de empates (1, 2, 2, 4) |
| `DENSE_RANK()` | Mismo rank para empates | No | No deja huecos (1, 2, 2, 3) |
| `NTILE(n)` | - | - | Divide filas en n grupos aproximadamente iguales |

En este laboratorio usamos `sales_historico` con equivalencias:
- `customer_id` -> `CustomerKey`
- `order_date` -> `OrderDate`
- `total_amount` -> `OrderQuantity`

In [11]:
SELECT TOP 50
    [CustomerKey] AS [customer_id],
    [OrderDate] AS [order_date],
    [OrderQuantity] AS [total_amount],
    ROW_NUMBER() OVER (
        PARTITION BY [CustomerKey]
        ORDER BY [OrderDate] DESC, [OrderNumber] DESC, [OrderLineItem] DESC
    ) AS [rn],
    RANK() OVER (
        PARTITION BY [CustomerKey]
        ORDER BY [OrderQuantity] DESC
    ) AS [rnk],
    DENSE_RANK() OVER (
        PARTITION BY [CustomerKey]
        ORDER BY [OrderQuantity] DESC
    ) AS [drnk],
    NTILE(4) OVER (ORDER BY [OrderQuantity] DESC) AS [quartile]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
ORDER BY [customer_id], [order_date] DESC;

Statement ID: {6FFFB9D0-F09D-4B1F-B6D4-31152B1FA3B3} | Query hash: 0x6DECAB6457C3453 | Distributed request ID: {656E3C21-EE04-495A-8BA4-9070E7CF636B}
(50 rows affected)

customer_id | order_date | total_amount | rn | rnk | drnk | quartile
------------+------------+--------------+----+-----+------+---------
11000       | 2016-11-04 | 2            | 4  | 1   | 1    | 1       
11000       | 2016-11-04 | 2            | 3  | 1   | 1    | 1       
11000       | 2016-11-04 | 1            | 2  | 4   | 2    | 3       
11000       | 2016-11-04 | 1            | 5  | 4   | 2    | 3       
11000       | 2016-11-04 | 1            | 1  | 4   | 2    | 3       
11000       | 2016-07-22 | 2            | 6  | 1   | 1    | 1       
11000       | 2016-07-22 | 1            | 7  | 4   | 2    | 3       
11001       | 2017-06-12 | 1            | 2  | 5   | 3    | 3       
11001       | 2017-06-12 | 1            | 4  | 5   | 3    | 3       
11001       | 2017-06-12 | 2            | 3  | 2   | 2    | 1       
11001       | 2017-06-12 | 1            | 1  | 5   | 3    | 3       
11001       | 2016-07-20 | 2            | 8  | 2   | 2    | 1       
11001       | 2016-07-20 | 2      

### Patron comun: ultima fila por grupo

`ROW_NUMBER()` + `PARTITION BY` es el patron estandar para deduplicacion y para obtener filas mas recientes por entidad.

In [13]:
WITH ranked AS (
    SELECT
        [CustomerKey] AS [customer_id],
        [OrderNumber] AS [order_id],
        [OrderDate] AS [order_date],
        [OrderQuantity] AS [total_amount],
        ROW_NUMBER() OVER (
            PARTITION BY [CustomerKey]
            ORDER BY [OrderDate] DESC, [OrderNumber] DESC, [OrderLineItem] DESC
        ) AS [rn]
    FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
)
SELECT TOP 20 *
FROM ranked
WHERE [rn] = 1
ORDER BY [customer_id];

Statement ID: {C5D76DF1-E0BA-4B14-9B17-914888F4402A} | Query hash: 0xB31687ECB0B8A96A | Distributed request ID: {57248FB2-EC21-421E-A775-DA3588646F4A}
(20 rows affected)

customer_id | order_id | order_date | total_amount | rn
------------+----------+------------+--------------+---
11000       | SO57418  | 2016-11-04 | 1            | 1 
11001       | SO72773  | 2017-06-12 | 1            | 1 
11002       | SO53237  | 2016-08-27 | 1            | 1 
11003       | SO57783  | 2016-11-11 | 2            | 1 
11004       | SO57293  | 2016-11-02 | 1            | 1 
11005       | SO57361  | 2016-11-03 | 1            | 1 
11007       | SO54705  | 2016-09-20 | 1            | 1 
11008       | SO53765  | 2016-09-03 | 1            | 1 
11009       | SO57736  | 2016-11-10 | 1            | 1 
11010       | SO58533  | 2016-11-24 | 3            | 1 
11011       | SO54706  | 2016-09-20 | 1            | 1 
11012       | SO68413  | 2017-04-17 | 2            | 1 
11013       | SO56137  | 2016-10-15 | 2            | 1 
11014       | SO57222  | 2016-11-01 | 4            | 1 
11015       | SO51520  | 2016-07-22 | 1            | 1 
11016       | SO52512  | 2016-08-13 | 2         

### Offset Functions: LAG y LEAD

| Funcion | Descripcion |
| --- | --- |
| `LAG(col, n, default)` | Valor de n filas antes de la fila actual |
| `LEAD(col, n, default)` | Valor de n filas despues de la fila actual |
| `FIRST_VALUE(col)` | Primer valor en el window frame |
| `LAST_VALUE(col)` | Ultimo valor en el window frame |

In [14]:
SELECT TOP 20
    [CustomerKey] AS [customer_id],
    [OrderDate] AS [order_date],
    [OrderQuantity] AS [total_amount],
    LAG([OrderQuantity], 1, 0) OVER (
        PARTITION BY [CustomerKey]
        ORDER BY [OrderDate], [OrderNumber], [OrderLineItem]
    ) AS [prev_amount],
    LEAD([OrderQuantity], 1, 0) OVER (
        PARTITION BY [CustomerKey]
        ORDER BY [OrderDate], [OrderNumber], [OrderLineItem]
    ) AS [next_amount],
    [OrderQuantity] - LAG([OrderQuantity], 1, 0) OVER (
        PARTITION BY [CustomerKey]
        ORDER BY [OrderDate], [OrderNumber], [OrderLineItem]
    ) AS [change_from_prev]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
ORDER BY [customer_id], [order_date];

Statement ID: {C520CF30-9129-454F-A8D3-5B966750A6E9} | Query hash: 0x32C94285DD96E760 | Distributed request ID: {C98EFFCC-472E-40CE-928E-A5D03D31D538}
(20 rows affected)

customer_id | order_date | total_amount | prev_amount | next_amount | change_from_prev
------------+------------+--------------+-------------+-------------+-----------------
11000       | 2016-07-22 | 1            | 0           | 2           | 1               
11000       | 2016-07-22 | 2            | 1           | 1           | 1               
11000       | 2016-11-04 | 1            | 2           | 2           | -1              
11000       | 2016-11-04 | 2            | 1           | 2           | 1               
11000       | 2016-11-04 | 2            | 2           | 1           | 0               
11000       | 2016-11-04 | 1            | 2           | 1           | -1              
11000       | 2016-11-04 | 1            | 1           | 0           | 0               
11001       | 2016-07-20 | 1            | 0           | 3           | 1               
11001       | 2016-07-20 | 3            | 1           | 2           | 2               
11001       | 2016-07-20 | 2            | 3

### FIRST_VALUE y LAST_VALUE

Regla clave: `LAST_VALUE` requiere un frame explicito que termine en `UNBOUNDED FOLLOWING`.
Si no se define, el frame por defecto termina en `CURRENT ROW` y `LAST_VALUE` devuelve el valor de la fila actual.

In [15]:
SELECT TOP 30
    [CustomerKey] AS [customer_id],
    [OrderDate] AS [order_date],
    [OrderQuantity] AS [total_amount],
    FIRST_VALUE([OrderQuantity]) OVER (
        PARTITION BY [CustomerKey]
        ORDER BY [OrderDate], [OrderNumber], [OrderLineItem]
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS [first_order_amount],
    LAST_VALUE([OrderQuantity]) OVER (
        PARTITION BY [CustomerKey]
        ORDER BY [OrderDate], [OrderNumber], [OrderLineItem]
        ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING
    ) AS [last_order_amount]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
ORDER BY [customer_id], [order_date];

Statement ID: {2CDAB932-9AD6-4F28-AEDC-65D50AFB3329} | Query hash: 0xF2A99C8CD52FD4CD | Distributed request ID: {557ADB27-645A-4421-B265-300ACFB2CDED}
(30 rows affected)

customer_id | order_date | total_amount | first_order_amount | last_order_amount
------------+------------+--------------+--------------------+------------------
11000       | 2016-07-22 | 2            | 1                  | 1                
11000       | 2016-07-22 | 1            | 1                  | 1                
11000       | 2016-11-04 | 1            | 1                  | 1                
11000       | 2016-11-04 | 1            | 1                  | 1                
11000       | 2016-11-04 | 2            | 1                  | 1                
11000       | 2016-11-04 | 2            | 1                  | 1                
11000       | 2016-11-04 | 1            | 1                  | 1                
11001       | 2016-07-20 | 2            | 1                  | 1                
11001       | 2016-07-20 | 1            | 1                  | 1                
11001       | 2016-07-20 | 1            | 1                  | 1                
11001       | 2016-07-20 | 2

### Agregados acumulados

Aqui primero agregamos por fecha para tener una serie diaria y luego aplicamos window functions.
`rolling_7day_sum` usa una ventana de 7 filas (6 anteriores + actual).

In [17]:
WITH daily_sales AS (
    SELECT
        [OrderDate] AS [order_date],
        SUM([OrderQuantity]) AS [total_amount]
    FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
    GROUP BY [OrderDate]
)
SELECT TOP 50
    [order_date],
    [total_amount],
    SUM([total_amount]) OVER (
        ORDER BY [order_date]
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS [running_total],
    AVG(CAST([total_amount] AS FLOAT)) OVER (
        ORDER BY [order_date]
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS [running_avg],
    COUNT(*) OVER (
        ORDER BY [order_date]
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS [running_count],
    SUM([total_amount]) OVER (
        ORDER BY [order_date]
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS [rolling_7day_sum]
FROM daily_sales
ORDER BY [order_date];

Statement ID: {DC9FCBEB-BB51-45DC-A2EA-015611ADF276} | Query hash: 0xDD7170727376959 | Distributed request ID: {F446604F-668A-4828-86D6-C29AF9EE4AF6}
(50 rows affected)

order_date | total_amount | running_total | running_avg        | running_count | rolling_7day_sum
-----------+--------------+---------------+--------------------+---------------+-----------------
2015-01-01 | 5            | 5             | 5                  | 1             | 5               
2015-01-02 | 4            | 9             | 4,5                | 2             | 9               
2015-01-03 | 8            | 17            | 5,666666666666667  | 3             | 17              
2015-01-04 | 7            | 24            | 6                  | 4             | 24              
2015-01-05 | 3            | 27            | 5,4                | 5             | 27              
2015-01-06 | 7            | 34            | 5,666666666666667  | 6             | 34              
2015-01-07 | 4            | 38            | 5,428571428571429  | 7             | 38              
2015-01-08 | 11           | 49            | 6,125              | 8             | 44              
2015-01-09 | 5      

### Exam tip

`ROW_NUMBER()` con `PARTITION BY` es el patron estandar para deduplicacion y para obtener filas mas recientes o Top N por grupo.
En escenarios de examen, primero identifica la particion (grupo) y luego el criterio de orden correcto (reciente, mayor monto, etc.).

## 8. CTEs y Subqueries

### Que es un CTE
Un `CTE` (Common Table Expression) es un resultado temporal nombrado que se define con `WITH` y existe solo durante la consulta que lo usa.
Sirve para dividir logica compleja en pasos legibles y reutilizables dentro de una sola sentencia SQL.

### Que es una Subquery
Una `subquery` es una consulta dentro de otra consulta.
Puede usarse en `SELECT`, `FROM` o `WHERE`, y puede ser no correlacionada o correlacionada.

Resumen rapido:
- `CTE`: prioriza legibilidad y organizacion por etapas.
- `Subquery`: prioriza resolver una condicion puntual dentro de otra consulta.

### Common Table Expression (CTE)

Ejemplo: total mensual y promedio movil de 3 meses sobre `sales_historico`.

In [18]:
WITH monthly_sales AS (
    SELECT
        YEAR([OrderDate]) AS [yr],
        MONTH([OrderDate]) AS [mo],
        SUM([OrderQuantity]) AS [monthly_total]
    FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
    GROUP BY YEAR([OrderDate]), MONTH([OrderDate])
)
SELECT
    [yr],
    [mo],
    [monthly_total],
    AVG(CAST([monthly_total] AS FLOAT)) OVER (
        ORDER BY [yr], [mo]
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS [moving_avg_3mo]
FROM monthly_sales
ORDER BY [yr], [mo];

Statement ID: {FA6C2D26-2D0C-4B09-9B95-783F06076463} | Query hash: 0x126303316C830548 | Distributed request ID: {C69D2B2D-136D-4907-8B9C-89085133E2BB}
(30 rows affected)

yr   | mo | monthly_total | moving_avg_3mo    
-----+----+---------------+-------------------
2015 | 1  | 216           | 216               
2015 | 2  | 181           | 198,5             
2015 | 3  | 220           | 205,66666666666666
2015 | 4  | 229           | 210               
2015 | 5  | 232           | 227               
2015 | 6  | 246           | 235,66666666666666
2015 | 7  | 287           | 255               
2015 | 8  | 321           | 284,6666666666667 
2015 | 9  | 218           | 275,3333333333333 
2015 | 10 | 238           | 259               
2015 | 11 | 219           | 225               
2015 | 12 | 357           | 271,3333333333333 
2016 | 1  | 267           | 281               
2016 | 2  | 292           | 305,3333333333333 
2016 | 3  | 291           | 283,3333333333333 
2016 | 4  | 330           | 304,3333333333333 
2016 | 5  | 367           | 329,3333333333333 
2016 | 6  | 341           | 346               
2016 | 7  | 2112          | 940               
2016 | 8  | 6

### Multiples CTEs

Se pueden encadenar varias CTEs para separar filtrado y agregacion en pasos claros.

In [19]:
WITH customers_cte AS (
    SELECT DISTINCT
        c.[CustomerKey] AS [customer_id]
    FROM [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_customers] AS c
    INNER JOIN [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico] AS s
        ON c.[CustomerKey] = s.[CustomerKey]
    WHERE s.[TerritoryKey] IN (1, 2, 3)
),
orders_cte AS (
    SELECT
        [CustomerKey] AS [customer_id],
        SUM([OrderQuantity]) AS [total_spent]
    FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
    GROUP BY [CustomerKey]
)
SELECT TOP 50
    c.[customer_id],
    o.[total_spent]
FROM customers_cte AS c
INNER JOIN orders_cte AS o
    ON c.[customer_id] = o.[customer_id]
ORDER BY o.[total_spent] DESC, c.[customer_id];

Statement ID: {E0D64C6E-D33F-447E-A01A-E9BDCBA19DD0} | Query hash: 0x47F0139FB6165D7A | Distributed request ID: {4FAD3437-9F06-4A38-8E4D-7D879013A42C}
(50 rows affected)

customer_id | total_spent
------------+------------
23696       | 20         
11325       | 19         
12533       | 19         
11263       | 17         
14337       | 17         
15827       | 17         
11014       | 16         
11829       | 16         
12142       | 16         
13431       | 16         
23002       | 16         
11264       | 15         
11529       | 15         
11535       | 15         
11725       | 15         
11953       | 15         
12198       | 15         
12204       | 15         
13884       | 15         
13894       | 15         
13898       | 15         
13977       | 15         
23732       | 15         
11063       | 14         
11153       | 14         
11254       | 14         
11282       | 14         
11673       | 14         
11838       | 14         
11843       | 14         
13274       | 14         
13468       | 14         
14240       | 14         
17463       | 14         
22742       | 14         
24734       | 14         
26520       

### Generación de Series de Fechas en Jupyter con SQL

En motores SQL tradicionales es posible utilizar **Recursive CTE** para generar secuencias de fechas o números. Sin embargo, en **Microsoft Fabric Warehouse / Synapse SQL**, las **consultas recursivas no están soportadas actualmente**.

Por esta razón, se emplea una alternativa moderna y eficiente mediante **GENERATE_SERIES**, que permite crear rangos secuenciales y convertirlos en fechas dinámicamente.

Esta metodología resulta útil para la creación de tablas calendario, análisis temporal de datos y proyectos de Business Intelligence desarrollados en Jupyter Notebook con Microsoft Fabric.

In [21]:
WITH date_bounds AS (
    SELECT
        MIN(CAST([OrderDate] AS date)) AS min_date,
        MAX(CAST([OrderDate] AS date)) AS max_date
    FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
)
SELECT
    DATEADD(DAY, gs.value, b.min_date) AS calendar_date,
    gs.value AS [level]
FROM date_bounds b
CROSS APPLY GENERATE_SERIES(
    0,
    DATEDIFF(DAY, b.min_date, b.max_date),
    1
) AS gs
ORDER BY calendar_date;

Statement ID: {F2773963-86B4-4E37-87E0-461E0C958766} | Query hash: 0xC8BE4EB3877F6D7C | Distributed request ID: {5FC22477-E90A-40E0-A932-3CFE7C1F6DEE}
(912 rows affected)

calendar_date | level
--------------+------
2015-01-01    | 0    
2015-01-02    | 1    
2015-01-03    | 2    
2015-01-04    | 3    
2015-01-05    | 4    
2015-01-06    | 5    
2015-01-07    | 6    
2015-01-08    | 7    
2015-01-09    | 8    
2015-01-10    | 9    
2015-01-11    | 10   
2015-01-12    | 11   
2015-01-13    | 12   
2015-01-14    | 13   
2015-01-15    | 14   
2015-01-16    | 15   
2015-01-17    | 16   
2015-01-18    | 17   
2015-01-19    | 18   
2015-01-20    | 19   
2015-01-21    | 20   
2015-01-22    | 21   
2015-01-23    | 22   
2015-01-24    | 23   
2015-01-25    | 24   
2015-01-26    | 25   
2015-01-27    | 26   
2015-01-28    | 27   
2015-01-29    | 28   
2015-01-30    | 29   
2015-01-31    | 30   
2015-02-01    | 31   
2015-02-02    | 32   
2015-02-03    | 33   
2015-02-04    | 34   
2015-02-05    | 35   
2015-02-06    | 36   
2015-02-07    | 37   
2015-02-08    | 38   
2015-02-09    | 39   
2015-02-10    | 40   
2015-02-11    | 41   
2015-02-12    | 42   
2015-02-13

### Correlated Subquery

La subquery correlacionada se ejecuta por cada fila de la consulta externa, porque depende de ella (`s.CustomerKey = c.CustomerKey`).

In [22]:
-- Clientes cuyo pedido maximo supera un umbral
SELECT TOP 30
    c.[CustomerKey] AS [customer_id]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_customers] AS c
WHERE (
    SELECT MAX(s.[OrderQuantity])
    FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico] AS s
    WHERE s.[CustomerKey] = c.[CustomerKey]
) > 3
ORDER BY [customer_id];

Statement ID: {23BC542F-A69F-4CD6-A7E7-E3684A54951E} | Query hash: 0x2E8431039FE48E47 | Distributed request ID: {B5738978-BD5B-46E3-8653-5C6274B1AE16}
(30 rows affected)

customer_id
-----------
11014      
11063      
11153      
11172      
11207      
11279      
11296      
11313      
11325      
11685      
11718      
11725      
11730      
11838      
11843      
11889      
11926      
11934      
11953      
11957      
11973      
12142      
12453      
12454      
12533      
12766      
12906      
12908      
12920      
12953      
(30 rows)

### EXISTS / NOT EXISTS

`EXISTS` verifica si hay al menos una coincidencia.
`NOT EXISTS` devuelve filas sin coincidencias en la subquery.

In [24]:
-- Clientes que han realizado al menos un pedido
SELECT TOP 30
    c.[CustomerKey] AS [customer_id]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_customers] AS c
WHERE EXISTS (
    SELECT 1
    FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico] AS s
    WHERE s.[CustomerKey] = c.[CustomerKey]
)
ORDER BY [customer_id];

-- Clientes sin pedidos
SELECT TOP 30
    c.[CustomerKey] AS [customer_id]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_customers] AS c
WHERE NOT EXISTS (
    SELECT 1
    FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico] AS s
    WHERE s.[CustomerKey] = c.[CustomerKey]
)
ORDER BY [customer_id];

Statement ID: {8D1E7A7C-80DA-4AA9-A1CA-BE205F756436} | Query hash: 0xBD17BCDC1C06203B | Distributed request ID: {3CEE1F54-C0DF-4A9C-9773-0DE6A2768E57}
(30 rows affected)
Statement ID: {4919470E-427C-43AA-99AD-8194158D6C6B} | Query hash: 0xC9B66CEAC1D81C90 | Distributed request ID: {5B1C7FDC-6EB6-45D2-830F-DC9275380801}
(30 rows affected)

customer_id
-----------
11000      
11001      
11002      
11003      
11004      
11005      
11007      
11008      
11009      
11010      
11011      
11012      
11013      
11014      
11015      
11016      
11017      
11018      
11019      
11020      
11021      
11022      
11023      
11024      
11025      
11026      
11027      
11028      
11029      
11030      
(30 rows)

customer_id
-----------
11051      
11133      
11349      
11435      
11441      
11552      
11585      
11595      
11597      
11733      
11786      
11927      
12233      
12313      
12375      
12405      
12417      
12443      
12456      
12478      
12529      
12542      
12792      
12801      
12876      
12890      
12911      
12974      
13033      
13094      
(30 rows)

### Exam tip

`EXISTS` suele ser mas eficiente que `IN` en subqueries grandes porque puede hacer short-circuit al encontrar la primera coincidencia.
Para deduplicacion, filtros de existencia y reglas de negocio binarias (tiene/no tiene), `EXISTS` y `NOT EXISTS` son patrones clave para examen y escenarios reales.

## 11. T-SQL especifico de Fabric

Esta seccion resume patrones clave de Fabric Warehouse para examen DP-600 y practica real.
Todos los ejemplos estan adaptados al contexto de `SEMANA2_WAREHOUSE` y `SEMANA2` que ya usas en este laboratorio.

### COPY INTO (ingesta de alto rendimiento)

Regla clave: `COPY INTO` es el comando principal para cargas masivas desde ADLS a Fabric Warehouse.
Usa comodines (`*`) para cargar multiples archivos en una sola operacion.

In [ ]:
-- Ejemplo 1: cargar CSV desde ADLS Gen2 hacia una tabla staging
-- Reemplaza URL y SAS antes de ejecutar
COPY INTO [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
FROM 'https://<storage-account>.dfs.core.windows.net/<container>/sales/*.csv'
WITH (
    FILE_TYPE = 'CSV',
    FIRSTROW = 2,
    FIELDTERMINATOR = ',',
    ROWTERMINATOR = '\n',
    CREDENTIAL = (
        IDENTITY = 'Shared Access Signature',
        SECRET = '<sas-token>'
    )
);

-- Ejemplo 2: cargar Parquet
COPY INTO [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
FROM 'https://<storage-account>.dfs.core.windows.net/<container>/sales/*.parquet'
WITH (
    FILE_TYPE = 'PARQUET',
    CREDENTIAL = (
        IDENTITY = 'Shared Access Signature',
        SECRET = '<sas-token>'
    )
);

Opciones frecuentes de `COPY INTO`:

| Opcion | Descripcion |
| --- | --- |
| `FILE_TYPE` | `CSV` o `PARQUET` |
| `FIRSTROW` | Fila inicial (para omitir encabezado en CSV) |
| `FIELDTERMINATOR` | Delimitador de columnas en CSV |
| `ROWTERMINATOR` | Delimitador de filas en CSV |
| `FIELDQUOTE` | Comilla para campos de texto en CSV |
| `ENCODING` | Codificacion del archivo (`UTF8`, etc.) |
| `CREDENTIAL` | Metodo de autenticacion para almacenamiento externo |

**Exam tip:** `COPY INTO` esta optimizado para cargas masivas y suele ser mas rapido que `INSERT` fila por fila.

### CREATE TABLE AS SELECT (CTAS)

Regla clave: CTAS materializa resultados en una sola operacion optimizada (crear + cargar).

In [ ]:
-- CTAS adaptado al laboratorio
IF OBJECT_ID('[adventureworks].[sales_historico_2017_ctas]', 'U') IS NOT NULL
BEGIN
    DROP TABLE [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico_2017_ctas];
END;

CREATE TABLE [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico_2017_ctas] AS
SELECT *
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico]
WHERE [OrderDate] >= '2017-01-01';

### Cross-Database Queries

Regla clave: en el mismo workspace puedes unir Warehouse y Lakehouse/SQL endpoint con nombres de tres partes: `database.schema.table`.

In [ ]:
-- Join entre Warehouse y SQL endpoint (mismo workspace)
SELECT TOP 50
    w.[OrderNumber] AS [order_id],
    w.[OrderQuantity] AS [total_amount],
    l.[ProductKey] AS [product_id]
FROM [SEMANA2_WAREHOUSE].[adventureworks].[sales_historico] AS w
INNER JOIN [SEMANA2].[adventureworks].[adventureworks_products] AS l
    ON w.[ProductKey] = l.[ProductKey]
ORDER BY w.[OrderNumber];

### SQL Analytics Endpoint

El SQL analytics endpoint expone tablas Delta del Lakehouse en modo T-SQL de solo lectura para datos.

| Caracteristica | Warehouse | SQL Analytics Endpoint |
| --- | --- | --- |
| DML (`INSERT`, `UPDATE`, `DELETE`) | Si | No (solo lectura) |
| DDL (`CREATE TABLE`) | Si | No |
| `CREATE VIEW` | Si | Si |
| `CREATE FUNCTION` | Si | Si |
| Cross-database queries | Si | Si |
| `SELECT` T-SQL | Si | Si |
| Seguridad (RLS, DDM) | Si | Si |
| Formato de almacenamiento | Delta Parquet | Delta Parquet |
| Modificacion de datos | T-SQL, pipelines | Spark, pipelines |

**Exam tip:** las tablas Delta creadas con Spark aparecen automaticamente en el endpoint SQL; no necesitas registrarlas manualmente.

### Dynamic Data Masking (DDM)

Paso 1: identifica columnas candidatas (email, telefono, etc.) en `adventureworks_customers`.

In [ ]:
SELECT
    [COLUMN_NAME],
    [DATA_TYPE]
FROM [SEMANA2_WAREHOUSE].INFORMATION_SCHEMA.COLUMNS
WHERE [TABLE_SCHEMA] = 'adventureworks'
  AND [TABLE_NAME] = 'adventureworks_customers'
ORDER BY [ORDINAL_POSITION];

In [ ]:
-- Reemplaza <email_column> y <phone_column> por columnas reales detectadas
ALTER TABLE [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_customers]
ALTER COLUMN [<email_column>] ADD MASKED WITH (FUNCTION = 'email()');

ALTER TABLE [SEMANA2_WAREHOUSE].[adventureworks].[adventureworks_customers]
ALTER COLUMN [<phone_column>] ADD MASKED WITH (FUNCTION = 'partial(0,"XXX-XXX-",4)');

### Row-Level Security (RLS)

Patron recomendado: usar una tabla de mapeo usuario-territorio y una funcion de predicado para filtrar filas automaticamente por `USER_NAME()`.

In [ ]:
-- 1) Crear esquema de seguridad si no existe
IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE [name] = 'security')
BEGIN
    EXEC('CREATE SCHEMA security');
END;

-- 2) Tabla de permisos por usuario y territorio
IF OBJECT_ID('[security].[user_territory_access]', 'U') IS NULL
BEGIN
    CREATE TABLE [SEMANA2_WAREHOUSE].[security].[user_territory_access] (
        [user_name] VARCHAR(256) NOT NULL,
        [TerritoryKey] INT NOT NULL,
        CONSTRAINT [PK_user_territory_access] PRIMARY KEY ([user_name], [TerritoryKey])
    );
END;

-- 3) Funcion de predicado para RLS
CREATE OR ALTER FUNCTION [security].[fn_territory_filter] (@TerritoryKey INT)
RETURNS TABLE
WITH SCHEMABINDING
AS
RETURN
    SELECT 1 AS [access]
    FROM [security].[user_territory_access] AS uta
    WHERE uta.[user_name] = USER_NAME()
      AND uta.[TerritoryKey] = @TerritoryKey;

-- 4) Crear policy solo si no existe
IF NOT EXISTS (
    SELECT 1
    FROM sys.security_policies
    WHERE [name] = 'territory_policy'
      AND [schema_id] = SCHEMA_ID('security')
)
BEGIN
    CREATE SECURITY POLICY [security].[territory_policy]
    ADD FILTER PREDICATE [security].[fn_territory_filter]([TerritoryKey])
    ON [adventureworks].[sales_historico]
    WITH (STATE = ON);
END;

-- 5) Ejemplo de grant inicial (ajusta el usuario real)
-- INSERT INTO [security].[user_territory_access] ([user_name], [TerritoryKey])
-- VALUES ('usuario@contoso.com', 1);

### Exam tip

`COPY INTO`, `CTAS`, cross-database query, RLS y DDM son temas de alta frecuencia en escenarios DP-600.
Piensa asi en examen:
1. Ingesta masiva -> `COPY INTO`.
2. Materializar transformaciones grandes -> `CTAS`.
3. Unir Warehouse + Lakehouse -> nombre de tres partes.
4. Proteger datos sensibles -> `MASKED` + `SECURITY POLICY`.